# Интервью-задание: GPU-парки OpenAI/Anthropic и маржа Sail Research по GLM-5.2

Данные собраны и расчёты выполнены **28 августа 2026 – 1 сентября 2026**.

Этот notebook воспроизводит все цифры, которые попали на два PDF-слайда (`slides.pdf`):

1. **Слайд 1** — сколько GPU у OpenAI и Anthropic, и как compute делится между обучением и инференсом.
2. **Слайд 2** — во сколько Sail Research обходится обслуживание GLM-5.2 на своей инфраструктуре и какая у них, по нашей оценке, валовая маржа при текущих ценах — с учётом того, кто такая Sail Research и как её цена соотносится с официальной ценой создателя модели (Z.ai).

Все внешние цифры — **оценки**, а не официально раскрытые данные (ни OpenAI, ни Anthropic, ни Sail Research не публикуют точные цифры по железу или себестоимости). Источники указаны в комментариях к каждому блоку и повторены на слайдах.


## Часть 1 — Данные и допущения (`calcs.py`)

Ниже — полный код с исходными числами и ссылками на источники. Он же используется модулем `make_slides.py` для генерации PDF.

In [1]:
"""
Interview task calculations
============================
Slide 1: OpenAI vs Anthropic GPU fleets, train/inference split
Slide 2: Sail Research GLM-5.2 margin (bottoms-up GPU-TCO model)

Data pulled: 2026-08-28. All external figures are estimates from public
reporting (companies do not disclose exact fleet sizes or infra costs).
Sources are listed inline and repeated on the slides.
"""

import json

# ---------------------------------------------------------------------
# SLIDE 1 — GPU fleets
# ---------------------------------------------------------------------

# OpenAI: Altman said "well over 1M GPUs" online by end of 2025 (own posts,
# reported widely, e.g. Tom's Hardware / Yahoo Finance, Jul 2025).
# Third-party datacenter tracker flopper.io (unofficial, extrapolated from
# public datacenter footprints) estimates ~1.4M physical GPUs / 5.7M
# H100-equivalents (H100e normalizes different chip generations to one
# comparable compute unit) as of mid-2026.
openai_gpus_physical_low = 1_000_000        # OpenAI's own "well over 1M" claim, EOY 2025
openai_gpus_physical_est = 1_400_000        # flopper.io tracker estimate, mid-2026
openai_h100e_est = 5_700_000                # flopper.io, H100-equivalent (accounts for newer/faster chips)

# Anthropic: does not disclose a GPU count; it runs on a mix of AWS
# Trainium, Google TPUs and Nvidia GPUs. Google deal: up to 1M TPUs,
# >1GW online in 2026 (Anthropic/Google press releases, Oct 2025 + Apr 2026).
# A second deal (Apr 2026) adds 3.5GW of TPU capacity from 2027.
# We convert power capacity to a rough GPU-equivalent using ~700W/accelerator
# (an analyst back-of-envelope conversion, TradingKey, Apr 2026) purely to
# make the two companies visually comparable on one slide — TPUs != GPUs,
# so this is explicitly flagged as an approximation on the slide.
anthropic_tpus_committed = 1_000_000        # "up to 1M TPUs" - Google/Anthropic Oct 2025 deal, contractual ceiling not necessarily deployed today
anthropic_gw_2026 = 1.0                     # >1GW TPU capacity online in 2026 (announced)
anthropic_gw_2027_add = 3.5                 # additional GW from 2027 (Apr 2026 deal)
watts_per_accelerator = 700                 # rough H100/H200-class TDP used for W->accelerator-count conversion
anthropic_accel_equiv_2026 = anthropic_gw_2026 * 1e9 / watts_per_accelerator
anthropic_accel_equiv_2027_total = (anthropic_gw_2026 + anthropic_gw_2027_add) * 1e9 / watts_per_accelerator

# Train vs inference split — no company publishes this. Two independent
# reference points, both flagged as estimates on the slide:
#  (a) Epoch AI / "Dean 2024"-style industry model: ~60-70% of *frontier lab*
#      compute went to training in 2025-2026, shifting toward inference
#      over the next few years (arXiv 2504.16138).
#  (b) Aggregator estimate (Lambda Finance, compiling SemiAnalysis/AWS/GCP
#      breadcrumbs) specifically for Anthropic: training 56%, inference 33%,
#      research 11% of compute *spend* (not compute-hours) — a single,
#      lower-confidence source, shown as a secondary data point only.
train_share_range = (0.60, 0.70)     # (a) industry-wide frontier-lab estimate, 2025-2026
inference_share_range = (0.30, 0.40)
anthropic_point_estimate = {"training": 0.56, "inference": 0.33, "research": 0.11}  # (b), single source, low confidence

# ---------------------------------------------------------------------
# SLIDE 2 — Sail Research GLM-5.2 margin model
# ---------------------------------------------------------------------

# --- Prices scraped from https://docs.sailresearch.com/pricing on 2026-08-28
# All figures: USD per 1,000,000 tokens.
glm_pricing = {
    "Default (ASAP)": {"input": 0.80, "cached": 0.16, "output": 3.00},
    "Balanced":        {"input": 0.50, "cached": 0.12, "output": 2.50},
    "Flex":            {"input": 0.40, "cached": 0.08, "output": 1.80},
}

# --- Model specs (public reporting on GLM-5.2, Zhipu/Z.ai, released 13 Jun 2026)
total_params = 744e9
active_params = 40e9          # active params per token (MoE) — consistent across sources
# Recommended deployment per vLLM's official recipe: FP8 checkpoint fits an
# 8xH200/8xH20 node (recipes.vllm.ai/zai-org/GLM-5.2). We model on 8xH200.
gpus_per_node = 8
h200_fp8_tflops_dense = 990   # NVIDIA published dense FP8 TFLOPS per H200 (no sparsity)

# --- Compute-bound cost model
# Standard rule of thumb used in industry cost models (e.g. SemiAnalysis):
#   FLOPs per generated token ~= 2 x active_parameters
# This gives a *lower-bound, compute-bound* estimate of GPU-time needed to
# serve one output token; real decode is often memory-bandwidth-bound at
# low batch sizes, but a commercial API provider batches heavily, which
# pushes realized throughput toward the compute-bound ceiling times an
# achieved "Model FLOPs Utilization" (MFU). We sweep MFU explicitly instead
# of guessing a single number.
flops_per_token = 2 * active_params

def node_tokens_per_sec(mfu):
    node_peak_flops = gpus_per_node * h200_fp8_tflops_dense * 1e12
    effective_flops = node_peak_flops * mfu
    return effective_flops / flops_per_token

def cost_per_million_output_tokens(mfu, usd_per_gpu_hour, overhead_multiplier=1.0):
    tps = node_tokens_per_sec(mfu)
    node_hours_per_million = 1e6 / tps / 3600
    gpu_hours_per_million = node_hours_per_million * gpus_per_node
    raw_cost = gpu_hours_per_million * usd_per_gpu_hour
    return raw_cost * overhead_multiplier

# --- Scenarios (bear / base / bull), each = (MFU, $/GPU-hr, overhead multiplier)
# GPU-hr rates from cloud-price surveys, Aug 2026 (Hyperbolic, GMI Cloud,
# Jarvislabs, getdeploying.com): on-demand H200 median ~$3.5-4.5/GPU-hr,
# specialised/reserved capacity as low as ~$1.5-2.5/GPU-hr.
# Overhead multiplier = fudge factor for real-world costs the pure FLOPs
# model ignores: sub-100% utilization, networking/storage, redundancy,
# non-serving staff & R&D allocated to inference infra. 1x = raw hardware
# only; 5x = generously loaded "fully-burdened" cost.
scenarios = {
    "Bear (thin/negative margin case)": dict(mfu=0.15, usd_per_gpu_hour=5.0, overhead_multiplier=5.0),
    "Base (reserved capacity, decent utilization)": dict(mfu=0.25, usd_per_gpu_hour=3.5, overhead_multiplier=2.0),
    "Bull (owned hardware, high utilization)": dict(mfu=0.40, usd_per_gpu_hour=2.0, overhead_multiplier=1.5),
}

# --- Sail Research — who they are (context for calibrating cost-model assumptions)
# Sail Research: VC-backed inference infra startup (Kleiner Perkins-led Series A +
# Sequoia-led seed, $80M total, $450M valuation, announced/emerged from stealth
# Jun 2026). Positions itself explicitly as throughput-over-latency infra for
# long-horizon agents, and makes specific efficiency claims in its own PR:
#   - "we carefully choose our chips, write custom inference engines, and run
#      a global controller that fully utilizes every computer in our fleet"
#     (Movva/CEO, via SiliconANGLE) -> a direct claim of high fleet utilization,
#     i.e. MFU near the top of our swept range, not the middle.
#   - claims "up to 10x lower cost per token than leading alternatives" and
#     topped the BrowseComp-Plus benchmark "at one-10th the inference cost of
#     rival services" (company PR / SiliconANGLE, Jun 2026).
# This doesn't prove a specific MFU number, but it's a reason to weight the
# Bull scenario as more representative of reality than Bear.
sail_context = {
    "funding_usd": 80_000_000,
    "valuation_usd": 450_000_000,
    "investors": ["Kleiner Perkins (Series A lead)", "Sequoia (Seed lead)", "Redpoint", "Theory Ventures", "Vine Ventures", "CRV"],
    "positioning": "throughput-over-latency inference infra for long-horizon AI agents",
    "self_claimed_advantage": "up to 10x lower cost per token than leading alternatives; custom inference engines; fleet run at high utilization by design",
}

# --- Cross-check: Z.ai's own first-party price for the same model, plus the
# cheapest observed third-party reseller rate, both checked 2026-08-28/09-01.
# Sources: VentureBeat (Jun 2026), layer3labs.io, orcarouter.ai, aipricing.guru
# (4 independent write-ups all agree on $1.40 / $4.40 for Z.ai's official API);
# OpenRouter model page for the cheapest third-party rate observed in the wild.
price_benchmarks_usd_per_1M = {
    "Z.ai (первоисточник модели, официальный API)": {"input": 1.40, "output": 4.40},
    "Sail Research, Default/ASAP":                   {"input": 0.80, "output": 3.00},
    "OpenRouter, самый дешёвый наблюдаемый reseller": {"input": 0.4875, "output": 1.56},
}
# Sail prices ~40% below the model creator's own list price, but well above
# the cheapest third-party reseller — i.e. squarely inside the competitive
# range for an open-weight model anyone can self-host, not an outlier discount.

results = {}
for name, params in scenarios.items():
    cost_1m_out = cost_per_million_output_tokens(**params)
    row = {"assumptions": params, "cost_per_1M_output_tokens_usd": round(cost_1m_out, 3), "margins": {}}
    for tier, prices in glm_pricing.items():
        margin = (prices["output"] - cost_1m_out) / prices["output"]
        row["margins"][tier] = round(margin * 100, 1)
    results[name] = row

if __name__ == "__main__":
    print("=== SLIDE 1 numbers ===")
    print("OpenAI physical GPUs (est.):", openai_gpus_physical_est)
    print("OpenAI H100-equivalents (est.):", openai_h100e_est)
    print("Anthropic accelerator-equivalent from committed power, 2026:", int(anthropic_accel_equiv_2026))
    print("Anthropic accelerator-equivalent from committed power, 2027+:", int(anthropic_accel_equiv_2027_total))
    print()
    print("=== SLIDE 2 numbers ===")
    print(json.dumps(results, indent=2))


print('OK: calcs.py loaded')


=== SLIDE 1 numbers ===
OpenAI physical GPUs (est.): 1400000
OpenAI H100-equivalents (est.): 5700000
Anthropic accelerator-equivalent from committed power, 2026: 1428571
Anthropic accelerator-equivalent from committed power, 2027+: 6428571

=== SLIDE 2 numbers ===
{
  "Bear (thin/negative margin case)": {
    "assumptions": {
      "mfu": 0.15,
      "usd_per_gpu_hour": 5.0,
      "overhead_multiplier": 5.0
    },
    "cost_per_1M_output_tokens_usd": 3.741,
    "margins": {
      "Default (ASAP)": -24.7,
      "Balanced": -49.6,
      "Flex": -107.8
    }
  },
  "Base (reserved capacity, decent utilization)": {
    "assumptions": {
      "mfu": 0.25,
      "usd_per_gpu_hour": 3.5,
      "overhead_multiplier": 2.0
    },
    "cost_per_1M_output_tokens_usd": 0.629,
    "margins": {
      "Default (ASAP)": 79.0,
      "Balanced": 74.9,
      "Flex": 65.1
    }
  },
  "Bull (owned hardware, high utilization)": {
    "assumptions": {
      "mfu": 0.4,
      "usd_per_gpu_hour": 2.0,
      "o

### Проверка чисел слайда 1

In [2]:
print("OpenAI, физич. GPU (оценка, flopper.io):", f"{openai_gpus_physical_est:,}")
print("OpenAI, H100-эквивалент (оценка, flopper.io):", f"{openai_h100e_est:,}")
print("Anthropic, accel-эквивалент по мощности 2026 (~1ГВт / 700Вт):", f"{int(anthropic_accel_equiv_2026):,}")
print("Anthropic, accel-эквивалент по мощности 2027+ (~4.5ГВт совокупно):", f"{int(anthropic_accel_equiv_2027_total):,}")
print()
print("Train/inference, отрасль (Epoch AI / industry model):", train_share_range, "/", inference_share_range)
print("Anthropic, оценка расходов на compute (Lambda Finance, единственный источник):", anthropic_point_estimate)


OpenAI, физич. GPU (оценка, flopper.io): 1,400,000
OpenAI, H100-эквивалент (оценка, flopper.io): 5,700,000
Anthropic, accel-эквивалент по мощности 2026 (~1ГВт / 700Вт): 1,428,571
Anthropic, accel-эквивалент по мощности 2027+ (~4.5ГВт совокупно): 6,428,571

Train/inference, отрасль (Epoch AI / industry model): (0.6, 0.7) / (0.3, 0.4)
Anthropic, оценка расходов на compute (Lambda Finance, единственный источник): {'training': 0.56, 'inference': 0.33, 'research': 0.11}


### Проверка чисел слайда 2 — модель себестоимости GLM-5.2

Логика (bottom-up, compute-bound):

1. GLM-5.2 — MoE, ~744B параметров всего, ~40B активных на токен.
2. Грубое правило (используется в отраслевых моделях типа SemiAnalysis): **FLOPs на сгенерированный токен ≈ 2 × активные параметры**.
3. Официальный рецепт деплоя (vLLM) — узел **8×H200** для FP8-чекпойнта.
4. Пиковая FP8-производительность H200 (без sparsity) — 990 TFLOPS/GPU (спецификация NVIDIA).
5. Реальная утилизация (MFU) неизвестна → берём диапазон 15–40%.
6. Стоимость GPU-часа неизвестна (свои датацентры vs аренда) → берём диапазон $2–5 (по обзорам рынка аренды H200, август 2026: медиана on-demand ~$3.5–4.5, специализированные/резервные предложения — от ~$1.5–2.5).
7. Overhead-множитель (сеть, недозагрузка, R&D, персонал) — 1.5×–5×, как отдельный поправочный коэффициент поверх «голой» FLOPs-модели.

Три сценария (bear/base/bull) комбинируют эти три параметра. Результат — оценка себестоимости на 1M **output**-токенов, которая сравнивается с ценой Sail Research.

In [3]:
import pandas as pd

rows = []
for scenario, r in results.items():
    row = {"Сценарий": scenario, **r["assumptions"], "Cost $/1M output tok": r["cost_per_1M_output_tokens_usd"]}
    row.update({f"Margin {t}": m for t, m in r["margins"].items()})
    rows.append(row)

df = pd.DataFrame(rows)
df


                                       Сценарий  ...  Margin Flex
0              Bear (thin/negative margin case)  ...       -107.8
1  Base (reserved capacity, decent utilization)  ...         65.1
2       Bull (owned hardware, high utilization)  ...         90.6

[3 rows x 8 columns]

### Кто такая Sail Research — почему это важно для выбора сценария

Первый черновик этого анализа считал маржу на чисто инженерной FLOPs-модели, без калибровки по тому, что сама компания заявляет о своей инфраструктуре. Это слепое пятно: Sail Research — не абстрактный реселлер модели, а венчурный стартап ($80M seed+A, $450M valuation, Kleiner Perkins/Sequoia, вышел из стелса в июне 2026), который **прямо позиционируется** как throughput-over-latency инфраструктура для длинных агентных задач и **заявляет**:

- «we carefully choose our chips, write custom inference engines, and run a global controller that fully utilizes every computer in our fleet» (CEO Neil Movva, via SiliconANGLE, июнь 2026) — прямая заявка на высокую утилизацию (MFU ближе к верхней границе диапазона, а не к середине);
- «up to 10x lower cost per token than leading alternatives» и топ на бенчмарке BrowseComp-Plus «at one-10th the inference cost of rival services» (компания, PR, июнь 2026).

Это не доказывает конкретное число MFU, но это довод в пользу того, что **Bull-сценарий ближе к реальности, чем Bear** — компания, чей основной питч инвесторам — операционная эффективность инференса, вряд ли работает при MFU 15%.

In [4]:
print("Sail Research — контекст:")
for k, v in sail_context.items():
    print(f"  {k}: {v}")


Sail Research — контекст:
  funding_usd: 80000000
  valuation_usd: 450000000
  investors: ['Kleiner Perkins (Series A lead)', 'Sequoia (Seed lead)', 'Redpoint', 'Theory Ventures', 'Vine Ventures', 'CRV']
  positioning: throughput-over-latency inference infra for long-horizon AI agents
  self_claimed_advantage: up to 10x lower cost per token than leading alternatives; custom inference engines; fleet run at high utilization by design


### Сверка цены Sail с официальной ценой создателя модели (Z.ai) и с самым дешёвым ресейлером

Ещё одна проверка, которой не было в первом черновике: сравнить цену Sail не только со своей себестоимостью, но и с рынком. GLM-5.2 — открытая модель (MIT), поэтому её продают десятки ресейлеров, включая самого Z.ai (создателя).

In [5]:
bench_df = pd.DataFrame(price_benchmarks_usd_per_1M).T
bench_df.columns = ["input $/1M", "output $/1M"]
bench_df


                                                input $/1M  output $/1M
Z.ai (первоисточник модели, официальный API)        1.4000         4.40
Sail Research, Default/ASAP                         0.8000         3.00
OpenRouter, самый дешёвый наблюдаемый reseller      0.4875         1.56

**Вывод из сверки:** цена Sail на output ($3.00) примерно на 30% ниже официальной цены Z.ai ($4.40), но заметно выше самого дешёвого наблюдаемого ресейлера на OpenRouter ($1.56). Это укладывается в обычный конкурентный разброс цен на открытую модель, которую может хостить кто угодно — не выглядит как демпинг ниже себестоимости, скорее как рыночная цена компании с реальным (заявленным) преимуществом по эффективности.

### Чувствительность: маржа как функция MFU и цены GPU-часа (при overhead ×2, тариф Default/ASAP)

Иллюстрирует, что главный рычаг — не какой-то один параметр, а их комбинация; при разумных (не крайних) предположениях маржа положительная в широком диапазоне.

In [6]:
import numpy as np

mfu_grid = np.linspace(0.10, 0.45, 8)
gpu_cost_grid = np.linspace(1.5, 5.5, 9)

heat = np.zeros((len(mfu_grid), len(gpu_cost_grid)))
for i, mfu in enumerate(mfu_grid):
    for j, gc in enumerate(gpu_cost_grid):
        cost = cost_per_million_output_tokens(mfu=mfu, usd_per_gpu_hour=gc, overhead_multiplier=2.0)
        margin = (glm_pricing["Default (ASAP)"]["output"] - cost) / glm_pricing["Default (ASAP)"]["output"]
        heat[i, j] = margin * 100

sens_df = pd.DataFrame(heat, index=[f"MFU {m:.0%}" for m in mfu_grid],
                        columns=[f"${g:.1f}/GPU-hr" for g in gpu_cost_grid])
sens_df.round(0)


         $1.5/GPU-hr  $2.0/GPU-hr  ...  $5.0/GPU-hr  $5.5/GPU-hr
MFU 10%         78.0         70.0  ...         25.0         18.0
MFU 15%         85.0         80.0  ...         50.0         45.0
MFU 20%         89.0         85.0  ...         63.0         59.0
MFU 25%         91.0         88.0  ...         70.0         67.0
MFU 30%         93.0         90.0  ...         75.0         73.0
MFU 35%         94.0         91.0  ...         79.0         76.0
MFU 40%         94.0         93.0  ...         81.0         79.0
MFU 45%         95.0         93.0  ...         83.0         82.0

[8 rows x 9 columns]

## Часть 2 — Генерация PDF-слайдов (`make_slides.py`)

Слайды рендерятся через matplotlib (16:9, 2 страницы в одном PDF: `slides.pdf`). Запуск ячейки ниже пересоздаёт файл `slides.pdf` из чисел выше.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from calcs import (
    openai_gpus_physical_est, openai_h100e_est,
    anthropic_accel_equiv_2026, anthropic_accel_equiv_2027_total,
    train_share_range, inference_share_range, anthropic_point_estimate,
    glm_pricing, results, sail_context, price_benchmarks_usd_per_1M,
)

plt.rcParams["font.family"] = "DejaVu Sans"
FIGSIZE = (13.333, 7.5)  # 16:9
BG = "#0f1520"
CARD = "#161d2b"
TEXT = "#e8ecf3"
MUTED = "#8b96ab"
ACCENT = "#5fb3ff"
ACCENT2 = "#ff9a5f"
GREEN = "#4fd18a"
RED = "#ff6b6b"

pdf = PdfPages("slides.pdf")

# ======================================================================
# SLIDE 1 — GPU fleets
# ======================================================================
fig = plt.figure(figsize=FIGSIZE, facecolor=BG)
fig.patch.set_facecolor(BG)

fig.text(0.045, 0.94, "Сколько GPU у OpenAI и Anthropic и как они делятся между обучением и инференсом",
          fontsize=17, color=TEXT, fontweight="bold", va="top")
fig.text(0.045, 0.885, "Данные актуальны на 28 августа 2026 · компании не раскрывают точные цифры — везде оценки, диапазоны показывают неопределённость",
          fontsize=9.5, color=MUTED, va="top")

# --- Left: bar chart of fleet size (H100-equivalent / accelerator-equivalent)
ax1 = fig.add_axes([0.05, 0.24, 0.44, 0.55])
ax1.set_facecolor(BG)
labels = ["OpenAI\nфизич. GPU", "OpenAI\nH100-экв.", "Anthropic\nэкв., ~1 ГВт\n(2026)", "Anthropic\nэкв., ~4.5 ГВт\n(2027+)"]
values = [openai_gpus_physical_est/1e6, openai_h100e_est/1e6, anthropic_accel_equiv_2026/1e6, anthropic_accel_equiv_2027_total/1e6]
colors = [ACCENT, ACCENT, ACCENT2, ACCENT2]
bars = ax1.bar(labels, values, color=colors, width=0.6, edgecolor="none")
for b, v in zip(bars, values):
    ax1.text(b.get_x()+b.get_width()/2, v+0.08, f"{v:.1f}M", ha="center", color=TEXT, fontsize=11, fontweight="bold")
ax1.set_ylabel("млн ускорителей (оценка)", color=MUTED, fontsize=9)
ax1.tick_params(colors=MUTED, labelsize=7.8)
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.spines["bottom"].set_visible(True)
ax1.spines["bottom"].set_color(MUTED)
ax1.set_ylim(0, 7.5)
ax1.set_title("Размер парка (оценки третьих сторон)", color=TEXT, fontsize=11, pad=10)

# --- Right: train/inference split
ax2 = fig.add_axes([0.545, 0.18, 0.41, 0.62])
ax2.set_facecolor(BG)
ax2.axis("off")
ax2.set_title("Train vs Inference — оценки отрасли", color=TEXT, fontsize=11, loc="left", pad=10)

# stacked bar for industry-wide range
y0 = 0.72
train_mid = np.mean(train_share_range)
ax2.barh([y0], [train_mid], color=ACCENT, height=0.16)
ax2.barh([y0], [1-train_mid], left=[train_mid], color=ACCENT2, height=0.16)
ax2.text(0.02, y0, f"Training ~{train_share_range[0]*100:.0f}–{train_share_range[1]*100:.0f}%", va="center", ha="left", color="#06111f", fontsize=9.5, fontweight="bold")
ax2.text(train_mid+0.02, y0, f"Inference ~{inference_share_range[0]*100:.0f}–{inference_share_range[1]*100:.0f}%", va="center", ha="left", color="#06111f", fontsize=9.5, fontweight="bold")
ax2.text(0, y0+0.13, "Frontier labs в целом (Epoch AI / industry model)", color=MUTED, fontsize=8.7)

# Anthropic point estimate (single lower-confidence source)
y1 = 0.38
segs = [("training",0.56,ACCENT), ("inference",0.33,ACCENT2), ("research",0.11,"#c9a4ff")]
x = 0
for name, val, col in segs:
    ax2.barh([y1], [val], left=[x], color=col, height=0.16)
    if val > 0.08:
        ax2.text(x+val/2, y1, f"{val*100:.0f}%", va="center", ha="center", color="#06111f", fontsize=9, fontweight="bold")
    x += val
ax2.text(0, y1+0.13, "Anthropic, оценка расходов на compute (1 источник — Lambda Finance,\nагрегирует SemiAnalysis/AWS/GCP; низкая достоверность)", color=MUTED, fontsize=8.2)

ax2.set_xlim(0,1)
ax2.set_ylim(0,1)

legend_y = 0.05
ax2.text(0, legend_y, "■", color=ACCENT, fontsize=11)
ax2.text(0.02, legend_y, "training", color=MUTED, fontsize=8.5, va="center")
ax2.text(0.13, legend_y, "■", color=ACCENT2, fontsize=11)
ax2.text(0.15, legend_y, "inference", color=MUTED, fontsize=8.5, va="center")
ax2.text(0.27, legend_y, "■", color="#c9a4ff", fontsize=11)
ax2.text(0.29, legend_y, "research (Anthropic-оценка)", color=MUTED, fontsize=8.5, va="center")

# Bottom takeaway box
fig.patches.append(plt.Rectangle((0.045, 0.095), 0.91, 0.10, transform=fig.transFigure,
                                  facecolor=CARD, edgecolor=ACCENT, linewidth=1.2))
fig.text(0.065, 0.178, "Вывод:", color=ACCENT, fontsize=10.5, fontweight="bold", va="top")
fig.text(0.065, 0.150,
         "OpenAI, по собственным заявлениям и оценкам трекеров, оперирует ~1–1.4 млн GPU (~5.7 млн H100-экв.); Anthropic не раскрывает парк GPU/TPU\n"
         "напрямую, но законтрактовала мощность, эквивалентную ~1.4–6.4 млн ускорителей к 2027 г. Доля обучения в compute у обеих компаний,\n"
         "по независимым оценкам, сегодня выше доли инференса (~60–70% vs ~30–40%), но, по прогнозам, будет снижаться к 2027–2028.",
         color=TEXT, fontsize=9.0, va="top")

fig.text(0.045, 0.03, "Источники: OpenAI/Altman (X, июль 2025); flopper.io datacenter tracker (не офиц.); Anthropic/Google press releases (окт. 2025, апр. 2026);\n"
                       "TradingKey (конверсия ГВт→GPU); Epoch AI (arXiv 2504.16138); Lambda Finance (агрегатор, вторичный источник для Anthropic split).",
         color=MUTED, fontsize=6.8, va="top")

pdf.savefig(fig, facecolor=BG)
plt.close(fig)

# ======================================================================
# SLIDE 2 — Sail Research GLM-5.2 margin
# ======================================================================
fig = plt.figure(figsize=FIGSIZE, facecolor=BG)
fig.patch.set_facecolor(BG)

fig.text(0.045, 0.975, "С какой маржой Sail Research продаёт GLM 5.2",
          fontsize=17, color=TEXT, fontweight="bold", va="top")
fig.text(0.045, 0.93, "Цены — docs.sailresearch.com/pricing, 28 авг 2026 · маржа — оценка по bottom-up GPU-TCO модели (не по раскрытым данным Sail Research)",
          fontsize=8.8, color=MUTED, va="top")

# --- Col 1: pricing table
ax1 = fig.add_axes([0.045, 0.635, 0.285, 0.245])
ax1.set_facecolor(BG)
ax1.axis("off")
ax1.set_title("Цены GLM-5.2 (Sail), $/1M ток.", color=TEXT, fontsize=9.7, loc="left")
tiers = list(glm_pricing.keys())
col_labels = ["Tier", "In", "Cache", "Out"]
cell_text = [[t.replace(" (ASAP)","").replace("Default","Default"), f"${glm_pricing[t]['input']:.2f}", f"${glm_pricing[t]['cached']:.2f}", f"${glm_pricing[t]['output']:.2f}"] for t in tiers]
table = ax1.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center",
                   colWidths=[0.36,0.22,0.22,0.22])
table.auto_set_font_size(False)
table.set_fontsize(8.6)
table.scale(1, 1.75)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#2a3345")
    if r == 0:
        cell.set_facecolor("#1f2937")
        cell.set_text_props(color=TEXT, fontweight="bold")
    else:
        cell.set_facecolor(CARD)
        cell.set_text_props(color=TEXT)

# --- Col 2: who is Sail Research
ax2 = fig.add_axes([0.365, 0.635, 0.285, 0.245])
ax2.set_facecolor(BG)
ax2.axis("off")
ax2.set_title("Кто такой Sail Research", color=TEXT, fontsize=9.7, loc="left")
ax2.text(0, 0.95, "$80M (seed+A) @ $450M valuation,", color=TEXT, fontsize=8.2, va="top")
ax2.text(0, 0.82, "Kleiner Perkins / Sequoia, июнь 2026", color=TEXT, fontsize=8.2, va="top")
ax2.text(0, 0.64, "Позиционирование: throughput-over-\nlatency инфра для агентов", color=TEXT, fontsize=8.2, va="top")
ax2.text(0, 0.36, "Собственная заявка (PR компании):", color=MUTED, fontsize=7.8, va="top")
ax2.text(0, 0.24, "«fully utilizes every computer in\nour fleet» + «до 10× дешевле\nконкурентов» — т.е. заявляют\nвысокую загрузку GPU по дизайну", color=MUTED, fontsize=7.8, va="top", style="italic")
ax2.set_xlim(0,1); ax2.set_ylim(0,1)

# --- Col 3: price cross-check vs model creator and cheapest reseller
ax3b = fig.add_axes([0.685, 0.635, 0.285, 0.245])
ax3b.set_facecolor(BG)
ax3b.axis("off")
ax3b.set_title("Сверка цены: кто сколько берёт", color=TEXT, fontsize=9.7, loc="left")
bench_labels = ["Z.ai\n(создатель\nмодели)", "Sail\nDefault", "OpenRouter\n(дешевле\nвсех)"]
bench_out = [price_benchmarks_usd_per_1M[k]["output"] for k in price_benchmarks_usd_per_1M]
bench_colors = [ACCENT2, ACCENT, "#4fd18a"]
bx = np.arange(3)
bars = ax3b.bar(bx, bench_out, color=bench_colors, width=0.55)
for b, v in zip(bars, bench_out):
    ax3b.text(b.get_x()+b.get_width()/2, v+0.08, f"${v:.2f}", ha="center", color=TEXT, fontsize=8, fontweight="bold")
ax3b.set_xticks(bx)
ax3b.set_xticklabels(bench_labels, color=MUTED, fontsize=7.3)
ax3b.set_ylabel("$/1M output ток.", color=MUTED, fontsize=7.5)
ax3b.tick_params(colors=MUTED, labelsize=7)
for spine in ax3b.spines.values():
    spine.set_visible(False)
ax3b.set_ylim(0, 5.2)

# --- model spec + cost model, one compact line each
fig.text(0.045, 0.605, "Модель: 744B всего / ~40B активных на токен (MoE), FP8 на узле 8×H200 (recipes.vllm.ai/zai-org/GLM-5.2)  ·  "
                       "Себестоимость: FLOPs/ток. ≈ 2×активные_парам., cost = GPU-часы × $/GPU-час × MFU⁻¹ × overhead",
         color=MUTED, fontsize=8.0, va="top")
fig.text(0.045, 0.578, "Три сценария различаются по MFU (15–40%), цене GPU-часа ($2–5) и overhead-множителю (1.5×–5×) — раскладка допущений в notebook.",
         color=MUTED, fontsize=8.0, va="top")

# --- Bottom: margin bar chart across scenarios and tiers
ax3 = fig.add_axes([0.045, 0.185, 0.91, 0.175])
ax3.set_facecolor(BG)
scenario_names = list(results.keys())
short_names = ["Bear — рыночная аренда,\nнизкая утилизация", "Base — резервная ёмкость,\nумеренная утилизация", "Bull — своё железо,\nвысокая утилизация\n(ближе к заявкам Sail)"]
x = np.arange(len(scenario_names))
width = 0.25
tier_colors = {"Default (ASAP)": ACCENT, "Balanced": "#8ecff5", "Flex": ACCENT2}
for i, tier in enumerate(glm_pricing.keys()):
    vals = [results[s]["margins"][tier] for s in scenario_names]
    bars = ax3.bar(x + (i-1)*width, vals, width, label=tier, color=tier_colors[tier])
    for b, v in zip(bars, vals):
        ax3.text(b.get_x()+b.get_width()/2, v + (2 if v>=0 else -6), f"{v:.0f}%", ha="center",
                  color=TEXT, fontsize=8, fontweight="bold")
ax3.axhline(0, color=MUTED, linewidth=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels(short_names, color=TEXT, fontsize=7.8)
ax3.set_ylabel("Валовая маржа, %", color=MUTED, fontsize=8.5)
ax3.tick_params(colors=MUTED, labelsize=8)
for spine in ax3.spines.values():
    spine.set_visible(False)
ax3.legend(loc="upper center", fontsize=8, facecolor=CARD, edgecolor="#2a3345", labelcolor=TEXT, ncol=3, framealpha=0.95)
ax3.set_ylim(-140, 175)
ax3.tick_params(axis="x", pad=10)

fig.patches.append(plt.Rectangle((0.045, 0.375), 0.91, 0.10, transform=fig.transFigure,
                                  facecolor=CARD, edgecolor=ACCENT, linewidth=1.2))
fig.text(0.065, 0.462, "Вывод:", color=ACCENT, fontsize=10.5, fontweight="bold", va="top")
fig.text(0.065, 0.436,
         "Даже в базовом сценарии маржа по всем тарифам положительная (~65–79%); в минус модель уходит только при совпадении худших предположений сразу по трём параметрам.\n"
         "Sail сама заявляет высокую загрузку фермы и специализированные inference-движки — это довод в пользу Base/Bull, а не Bear.\n"
         "Цена Sail ($3.00) на ~30% ниже официальной цены Z.ai ($4.40) за output, но выше минимальной рыночной ($1.56) — конкурентный reseller-прайсинг, не демпинг.",
         color=TEXT, fontsize=8.0, va="top")

fig.text(0.045, 0.03, "Источники: docs.sailresearch.com/pricing (28.08.2026); recipes.vllm.ai/zai-org/GLM-5.2; NVIDIA H200 datasheet (FP8 TFLOPS); Hyperbolic/GMI Cloud/Jarvislabs/getdeploying.com (аренда H200, авг. 2026);\n"
                       "SiliconANGLE / Fortune / AI Weekly (о Sail Research, июнь 2026); VentureBeat / layer3labs.io / orcarouter.ai / aipricing.guru (цена Z.ai); OpenRouter (минимальная цена reseller'ов). Полный расчёт — в notebook.",
         color=MUTED, fontsize=6.6, va="top")

pdf.savefig(fig, facecolor=BG)
plt.close(fig)

pdf.close()
print("saved slides.pdf")


saved slides.pdf


## Ограничения этой оценки

- **Слайд 1**: ни OpenAI, ни Anthropic не публикуют точный состав парка (кол-во GPU/TPU, распределение по поколениям чипов) или разбивку compute на train/inference. Все цифры — экстраполяции третьих сторон (datacenter-трекеры, конверсия объявленной мощности в ГВт в число ускорителей, отраслевые модели) и должны восприниматься как ориентир порядка величины, а не точное число.
- **Слайд 2**: у нас нет прямых данных о фактической утилизации GPU у Sail Research, их реальной цене за GPU-час (own hardware vs contract vs on-demand) или накладных расходах. Модель считает только *инференс-compute* по generation-токенам (FLOPs-правило), не включает: стоимость обучения/файнтюна модели третьей стороной (Zhipu, не Sail Research), затраты на prefill/input-токены отдельно (они дешевле per-token и по цене, и по себестоимости, поэтому исключение их из модели консервативно занижает реальную маржу), и любые немаржинальные бизнес-расходы (продажи, поддержка, R&D помимо инфраструктуры). Калибровка сценариев по собственным публичным заявлениям Sail Research (см. выше) снижает, но не устраняет эту неопределённость — заявления компании о своей эффективности не независимо верифицированы.